# Chapter 4 — Donut characterization

This notebook only reads saved experiment records. It does not load or execute a model. Missing records leave tables and plots empty until experiments are run.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("../results/ch04")


def load(name):
    path = RESULTS / name
    return json.loads(path.read_text()) if path.exists() else None


inventory = load("inventory.json")
inference = load("inference-resolution-length.json")
inference_batch = load("inference-batch-scaling.json")
training = load("training-resolution-length.json")
training_batch = load("training-batch-scaling.json")
components = load("components-1920x1440.json")
real_profile = load("br-real-profile.json")

## Static architecture inventory

In [ ]:
if inventory:
    params = inventory["measurements"]["parameters"]
    parameter_table = pd.DataFrame(
        {
            "component": [
                "complete model",
                "encoder",
                "decoder",
                "vocabulary projection (tied)",
            ],
            "parameters": [
                params["model"],
                params["encoder"],
                params["decoder"],
                params["lm_head"],
            ],
        }
    )
    display(parameter_table)

    rows = []
    for resolution in inventory["measurements"]["resolutions"]:
        for stage in resolution["stages"]:
            rows.append(
                {"resolution": "x".join(map(str, resolution["image"])), **stage}
            )
    stage_table = pd.DataFrame(rows)
    display(stage_table)
else:
    print("inventory.json is missing")

## Controlled inference and training

In [ ]:
def controlled_frame(record, kind):
    if not record:
        return pd.DataFrame()
    rows = []
    for item in record["measurements"]["records"]:
        row = {
            "status": item["status"],
            "resolution": "x".join(map(str, item["resolution"])),
            "batch_size": item["batch_size"],
        }
        if item["status"] == "ok":
            if kind == "inference":
                row.update(
                    {
                        "sequence_length": item["output_length"],
                        "visual_tokens": item["visual_tokens"],
                        "encoder_ms": item["encoder"]["median_ms"],
                        "decoder_ms": item["decoder"]["median_ms"],
                        "total_ms": item["full"]["median_ms"],
                        "peak_mb": item["memory"]["full"]["peak_allocated_mb"],
                    }
                )
            else:
                row.update(
                    {
                        "sequence_length": item["target_length"],
                        "forward_ms": item["summary"]["forward_ms"]["median"],
                        "backward_ms": item["summary"]["backward_ms"]["median"],
                        "total_ms": item["summary"]["total_ms"]["median"],
                        "peak_mb": item["memory"]["peak_allocated_mb"],
                    }
                )
        rows.append(row)
    return pd.DataFrame(rows)


inference_df = controlled_frame(inference, "inference")
training_df = controlled_frame(training, "training")
display(inference_df)
display(training_df)

In [ ]:
if not inference_df.empty and not training_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for resolution, group in inference_df.query("status == 'ok'").groupby("resolution"):
        axes[0].plot(
            group.sequence_length, group.decoder_ms, marker="o", label=resolution
        )
    axes[0].set(xlabel="generated tokens", ylabel="decoder latency (ms)")
    axes[0].legend()

    fixed = inference_df.query("status == 'ok' and sequence_length == 56")
    axes[1].bar(fixed.resolution, fixed.encoder_ms)
    axes[1].set(xlabel="resolution", ylabel="encoder latency (ms)")

    fixed_train = training_df.query("status == 'ok' and sequence_length == 56")
    axes[2].bar(fixed_train.resolution, fixed_train.total_ms)
    axes[2].set(xlabel="resolution", ylabel="training-step latency (ms)")
    plt.tight_layout()
else:
    print("Controlled result records are missing")

## Component attribution

In [ ]:
if components:
    encoder_components = pd.DataFrame(
        components["measurements"]["encoder_components"]
    ).T
    decoder_components = pd.DataFrame(
        components["measurements"]["decoder_components"]
    ).T
    display(encoder_components)
    display(decoder_components)
else:
    print("components-1920x1440.json is missing")

## Final quality–cost table

In [ ]:
summary_rows = [
    {
        "dataset": "BR",
        "resolution": "1920x1440",
        "fine_tuning_s": None,
        "micro_f1": None,
        "macro_f1": None,
        "latency_p50_ms": None,
        "latency_p95_ms": None,
    },
    {
        "dataset": "KPID",
        "resolution": None,
        "fine_tuning_s": None,
        "micro_f1": None,
        "macro_f1": None,
        "latency_p50_ms": None,
        "latency_p95_ms": None,
    },
    {
        "dataset": "KPD",
        "resolution": None,
        "fine_tuning_s": None,
        "micro_f1": None,
        "macro_f1": None,
        "latency_p50_ms": None,
        "latency_p95_ms": None,
    },
]
if real_profile:
    br = summary_rows[0]
    measurements = real_profile["measurements"]
    br["micro_f1"] = measurements["strict"]["micro"]["f1"]
    br["macro_f1"] = measurements["strict"]["macro_field_f1"]
    br["latency_p50_ms"] = measurements["profile"]["total_ms"]["median_ms"]
    br["latency_p95_ms"] = measurements["profile"]["total_ms"]["p95_ms"]
training_record = Path("../checkpoints/br-1920x1440/train.json")
if training_record.exists():
    trained = json.loads(training_record.read_text())
    summary_rows[0]["fine_tuning_s"] = trained["measurements"]["fine_tuning_seconds"]
summary_table = pd.DataFrame(summary_rows)
display(summary_table)